## Tries
Tries are trees used for efficient data retrieval. Every node of Trie consists of multiple branches. Each branch represents a possible character of keys. We need to mark the last node of every key as end of word node.

In [1]:
class TrieNode {
    public TrieNode[] nodes = new TrieNode[26];
    public boolean isWord;
}

The following operations are provided:
- `insert(String word)` adds a string to the Trie
- `search(String word)` checks if a string is present in Trie
- `startsWith(String prefix)` checks if Trie contains a string that starts with given prefix

In [3]:
class Trie {
    TrieNode root = new TrieNode();

    // O(m) where m = length of word
    public void insert(String word) {
        TrieNode current = root;
        for (char c : word.toCharArray()) {
            int index = c - 'a';
            if (current.nodes[index] == null) {
                current.nodes[index] = new TrieNode();
            }

            current = current.nodes[index];
        }
        current.isWord = true;
    }

    // O(m) where m = length of word
    public boolean search(String word) {
        TrieNode current = root;
        for (char c : word.toCharArray()) {
            int index = c - 'a';
            if (current.nodes[index] == null) {
                return false;
            }
            current = current.nodes[index];
        }
        return current.isWord;
    }

    // O(m) where m = length of prefix
    public boolean startsWith(String prefix) {
        TrieNode current = root;
        for (char c : prefix.toCharArray()) {
            int index = c - 'a';
            if (current.nodes[index] == null) {
                return false;
            }
            current = current.nodes[index];
        }
        return true;
    }
}

Trie trie = new Trie();
trie.insert("apple");
trie.insert("banana");
System.out.println(trie.search("banana"));
System.out.println(trie.search("appl"));
System.out.println(trie.startsWith("appl"));

true
false
true


## Problems
**Q 1:** Lets say the search word can have '.' characters where '.' represents a single character. How do we search then?  
**Answer:** we can use a queue and add candidate nodes there.

In [ ]:
record TrieNodeIndex(TrieNode node, int index) {}
public boolean searchWildCard(String word) {
    Queue<TrieNodeIndex> queue = new ArrayDeque<>();
    queue.add(new TrieNodeIndex(root, 0));

    while (!queue.isEmpty()) {
        TrieNodeIndex nodeIndex = queue.poll();

        TrieNode node = nodeIndex.node();
        int index = nodeIndex.index();

        if (index == word.length()) {
            if (node.isWord) {
                return true;
            }

            continue;
        }
        char term = word.charAt(index);

        if (term == '.') {
            for (TrieNode temp : node.nodes) {
                if (temp != null) {
                    queue.add(new TrieNodeIndex(temp, index + 1));
                }
            }
        } else {
            int j = term - 'a';
            if (node.nodes[j] != null) {
                queue.add(new TrieNodeIndex(node.nodes[j], index + 1));
            }
        }
    }

    return false;
}

[LeetCode 211](https://leetcode.com/problems/design-add-and-search-words-data-structure)

**Q 2:** Trie datastructure is an excellent choice for building autocomplete. As an example, when we type do, we should get results as dog, dove, dome. When we type e, eagle should be returned. This is provided that we previously inserted those terms into the trie.

In [ ]:
class Trie { 
    public TrieNode root = new TrieNode(); 
    
    public void insert(String word) { 
        TrieNode current = root; 
        for (char c : word.toCharArray()) { 
            current = current.nodes.computeIfAbsent(c, k -> new TrieNode()); 
        } 
        
        current.isWord = true; 
        current.frequency++; 
    } 
    
    public boolean search(String word) { 
        TrieNode current = root; 
        for (char c : word.toCharArray()) { 
            current = current.nodes.get(c); 
            if (current == null) { 
                return false; 
            } 
        } 
        
        return current.isWord; 
    } 
    
    public boolean startsWith(String prefix) { 
        TrieNode current = root; 
        for (char c : prefix.toCharArray()) { 
            current = current.nodes.get(c); 
            if (current == null) { 
                return false; 
            } 
        } 
        
        return true; 
    } 
    
    public List<String> autocomplete(String prefix, int n) { 
        TrieNode current = root; 
        
        // Reach prefix node 
        List<String> result = new ArrayList<>(); 
        for (char c : prefix.toCharArray()) { 
            current = current.nodes.get(c); 
            if (current == null) { 
                return result; 
            } 
        } 
        
        // Do dfs from the prefix node 
        StringBuilder part = new StringBuilder(prefix); 
        List<Suggestion> suggestions = new ArrayList<>(); 
        explore(current, part, suggestions); 
        suggestions.stream() 
            .sorted( 
                Comparator.comparingInt(Suggestion::frequency) 
                .reversed() 
                .thenComparing(Suggestion::term) ) 
            .limit(n) 
            .map(Suggestion::term) 
            .forEach(result::add); 
        return result; 
    } 
    
    private void explore(TrieNode node, StringBuilder part, List<Suggestion> result) { 
        if (node.isWord) { 
            result.add(new Suggestion(part.toString(), node.frequency)); 
        } 
        
        for (Map.Entry<Character, TrieNode> entry : node.nodes.entrySet()) { 
            part.append(entry.getKey()); 
            explore(entry.getValue(), part, result); 
            part.deleteCharAt(part.length() - 1); 
        } 
    } 
    
    private static record Suggestion(String term, int frequency) { } 
    
    private static class TrieNode { 
        public Map<Character, TrieNode> nodes = new HashMap<>(); 
        public boolean isWord; 
        public int frequency; 
    }
} 